# Creating a Rope Bridge in C++ with Unreal Engine 5

## Introduction

In this document, I will detail the creation of a rope bridge subjected to physics. Additionally, the bridge will be generated procedurally.

## Description

To create the bridge, we first need to create a C++ class that inherits from the AActor class (the base class for objects in Unreal Engine). The bridge will consist of a SplinesComponent (a spline is a parametric curve) that will be generated procedurally using [catenary functions](https://en.wikipedia.org/wiki/Catenary) (a mathematical function that models the shape of a hanging chain). The spline will then be used to generate the planks of the bridge, which can have random offsets to give a more natural look.

## Steps

### 1 - Project Creation

To start, create a new Unreal Engine 5 project. Launch the software and click on "New Project". Then, choose the "Blank" template and select C++ as the language. Other settings can be left as default.

<img src="Images/img1.png" alt="Create a new project" style="width: 300px;"/>

### 2 - Creating the Bridge Class

To create the bridge class, right-click in the "Source" folder of the project and select "New C++ Class". Then, choose "Actor" as the parent class. The class name does not matter, but for this example, I will call it "RopesBridge".

<img src="Images/img2.png" alt="Create a new class" style="width: 200px;"/>
<br>
<img src="Images/img3.png" alt="Create a new class" style="width: 200px;"/>

You can immediately delete the BeginPlay and Tick functions from the class since we won't be using them.

### 3 - Adding the FCurveParams Structure

When you open the class, you will see that two files have been created: RopesBridge.h and RopesBridge.cpp. The .h file contains the class declaration (variables and function names) and the .cpp file contains the function implementations (the code that will be executed).

In the .h file, we will add an FCurveParams structure that will contain the curve parameters. This structure will include the following variables:
- CatenaryAlpha, a float representing the alpha parameter of the catenary function. The default value is 5.0f.
- CatenaryBeta, a float representing the beta parameter of the catenary function. The default value is 17.5f.
- CatenaryBetaOffset, a float representing the offset of the beta parameter of the catenary function. The default value is 0.0f.
- SplineSubdivisions, an int32 representing the number of spline subdivisions. The default value is 25.

This structure is placed just before the declaration of the RopesBridge class in the .h file and is declared as follows:

```c++
USTRUCT(BlueprintType)
struct FCurveParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The alpha parameter of the catenary equation. Defines the shape of the curve."))
	float CatenaryAlpha = 5.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The beta parameter of the catenary equation. Defines the portion of the curve used for the bridge."))
	float CatenaryBeta = 17.5f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The offset of the catenary equation. Defines the horizontal shift of the curve."))
	float CatenaryBetaOffset = 0.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties")
	int SplineSubdivisions = 25;
};
```

### 4 - Adding the SplinesComponent, GenerateSpline Function, and Associated Variables

In the .h file, add the following components in the private section of the RopesBridge class:

```c++
UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
USceneComponent* Root;

UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
USplineComponent* Spline;
```

Then add the following variables in the protected section of the RopesBridge class:

```c++
UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
float BridgeLength = 1000.0f;

UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
FCurveParams CurveParams;
```

Finally, add the GenerateSpline function in the protected section of the RopesBridge class and the CalculateChainHeight and CalculateChainTangent functions in the private section of the RopesBridge class. The GenerateSpline function will generate the spline using catenary functions, and the CalculateChainHeight and CalculateChainTangent functions will calculate the height and tangent of the catenary curve at a given point in space.

```c++
protected:
    UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
    void GenerateSplines();

private:
    UFUNCTION(BlueprintCallable, Category = "Spline generation")
	float CalculateChainHeight(float X);

	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	FVector CalculateChainTangent(FVector Previous, FVector Current, FVector Next);
```

At this point, the .h file should look like this:

```c++
#pragma once

#include "CoreMinimal.h"
#include "GameFramework/Actor.h"
#include "RopeBridge.generated.h"

class USplineComponent;

USTRUCT(BlueprintType)
struct FCurveParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The alpha parameter of the catenary equation. Defines the shape of the curve."))
	float CatenaryAlpha = 5.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The beta parameter of the catenary equation. Defines the portion of the curve used for the bridge."))
	float CatenaryBeta = 17.5f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The offset of the catenary equation. Defines the horizontal shift of the curve."))
	float CatenaryBetaOffset = 0.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties")
	int SplineSubdivisions = 25;
};

UCLASS()
class ROPESBRIDGE_API ARopeBridge : public AActor
{
	GENERATED_BODY()
	
	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USceneComponent* Root;

	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USplineComponent* Spline;
	
public:	
	ARopeBridge();

protected:
	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	float BridgeLength = 1000.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	FCurveParams CurveParams;

	UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
	void GenerateSplines();

private:
	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	float CalculateChainHeight(float X);

	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	FVector CalculateChainTangent(FVector Previous, FVector Current, FVector Next);
};
```

### 5 - Implementing the Class Constructor

The class constructor is a function called when the object is created. In the .cpp file, implement the constructor of the RopesBridge class. Create the Root and Spline components and attach them to the object.

```c++
ARopesBridge::ARopesBridge()
{
	PrimaryActorTick.bCanEverTick = false;

	Root = CreateDefaultSubobject<USceneComponent>(TEXT("Root"));
	RootComponent = Root;

	Spline = CreateDefaultSubobject<USplineComponent>(TEXT("Spline"));
	Spline->SetupAttachment(Root);
}
```

### 6 - Implementing the GenerateSpline Function and Associated Functions

In the .cpp file, implement the GenerateSpline, CalculateChainHeight, and CalculateChainTangent functions. The GenerateSpline function will generate the spline using catenary functions, and the CalculateChainHeight and CalculateChainTangent functions will calculate the height and tangent of the catenary curve at a given point in space.

```c++
void ARopeBridge::GenerateSplines()
{
	Spline->ClearSplinePoints(false);
		
	float HeightOffset = CalculateChainHeight(0.0f);
	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Location = FVector(BridgeLength * i / (CurveParams.SplineSubdivisions - 1), 0.0f, CalculateChainHeight(i / (float)(CurveParams.SplineSubdivisions - 1)) - HeightOffset);
		Spline->AddSplinePoint(Location, ESplineCoordinateSpace::Local, false);
	}

	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Tangent = FVector(0.0f, 0.0f, 0.0f);

		if (i != 0 && i < CurveParams.SplineSubdivisions - 1)
		{
			FVector PreviousLocation = Spline->GetLocationAtSplinePoint(i - 1, ESplineCoordinateSpace::

Local);
			FVector Location = Spline->GetLocationAtSplinePoint(i, ESplineCoordinateSpace::Local);
			FVector NextLocation = Spline->GetLocationAtSplinePoint(i + 1, ESplineCoordinateSpace::Local);

			Tangent = CalculateChainTangent(PreviousLocation, Location, NextLocation);
		}

		Spline->SetTangentAtSplinePoint(i, Tangent, ESplineCoordinateSpace::Local, false);
	}

	Spline->UpdateSpline();
}

float ARopeBridge::CalculateChainHeight(float X)
{
	return (CurveParams.CatenaryAlpha * cosh(X * CurveParams.CatenaryBeta) + CurveParams.CatenaryBetaOffset);
}

FVector ARopeBridge::CalculateChainTangent(FVector Previous, FVector Current, FVector Next)
{
	FVector Tangent = (Next - Previous).GetSafeNormal();
	return Tangent;
}
```

At this point, the .cpp file should look like this:

```c++
#include "RopesBridge.h"
#include "Components/SplineComponent.h"
#include "Math/UnrealMathUtility.h"

ARopesBridge::ARopesBridge()
{
	PrimaryActorTick.bCanEverTick = false;

	Root = CreateDefaultSubobject<USceneComponent>(TEXT("Root"));
	RootComponent = Root;

	Spline = CreateDefaultSubobject<USplineComponent>(TEXT("Spline"));
	Spline->SetupAttachment(Root);
}

void ARopesBridge::GenerateSplines()
{
	Spline->ClearSplinePoints(false);
		
	float HeightOffset = CalculateChainHeight(0.0f);
	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Location = FVector(BridgeLength * i / (CurveParams.SplineSubdivisions - 1), 0.0f, CalculateChainHeight(i / (float)(CurveParams.SplineSubdivisions - 1)) - HeightOffset);
		Spline->AddSplinePoint(Location, ESplineCoordinateSpace::Local, false);
	}

	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Tangent = FVector(0.0f, 0.0f, 0.0f);

		if (i != 0 && i < CurveParams.SplineSubdivisions - 1)
		{
			FVector PreviousLocation = Spline->GetLocationAtSplinePoint(i - 1, ESplineCoordinateSpace::Local);
			FVector Location = Spline->GetLocationAtSplinePoint(i, ESplineCoordinateSpace::Local);
			FVector NextLocation = Spline->GetLocationAtSplinePoint(i + 1, ESplineCoordinateSpace::Local);

			Tangent = CalculateChainTangent(PreviousLocation, Location, NextLocation);
		}

		Spline->SetTangentAtSplinePoint(i, Tangent, ESplineCoordinateSpace::Local, false);
	}

	Spline->UpdateSpline();
}

float ARopesBridge::CalculateChainHeight(float X)
{
	return (CurveParams.CatenaryAlpha * cosh(X * CurveParams.CatenaryBeta) + CurveParams.CatenaryBetaOffset);
}

FVector ARopesBridge::CalculateChainTangent(FVector Previous, FVector Current, FVector Next)
{
	FVector Tangent = (Next - Previous).GetSafeNormal();
	return Tangent;
}
```

### 7 - Adding and Configuring the Plank Meshes

Now that the spline has been generated, we can add planks along the spline to create the bridge. In the .h file, add the following variables in the private section of the RopesBridge class:

```c++
UPROPERTY(EditAnywhere, Category = "Bridge generation")
UStaticMesh* PlankMesh;

UPROPERTY(EditAnywhere, Category = "Bridge generation")
int32 PlankCount = 25;

UPROPERTY(EditAnywhere, Category = "Bridge generation")
float PlankWidth = 100.0f;

UPROPERTY(EditAnywhere, Category = "Bridge generation")
float PlankRandomOffset = 50.0f;
```

Then, in the .cpp file, add the following code to the GenerateSplines function to create the planks along the spline:

```c++
for (int i = 0; i < PlankCount; i++)
{
	float Distance = BridgeLength * i / (PlankCount - 1);
	FVector Location = Spline->GetLocationAtDistanceAlongSpline(Distance, ESplineCoordinateSpace::World);
	FRotator Rotation = Spline->GetRotationAtDistanceAlongSpline(Distance, ESplineCoordinateSpace::World);

	float RandomOffset = FMath::RandRange(-PlankRandomOffset, PlankRandomOffset);
	FVector Offset = FVector(0.0f, RandomOffset, 0.0f);

	FActorSpawnParameters SpawnParams;
	AStaticMeshActor* Plank = GetWorld()->SpawnActor<AStaticMeshActor>(Location + Offset, Rotation, SpawnParams);
	Plank->GetStaticMeshComponent()->SetStaticMesh(PlankMesh);
	Plank->GetStaticMeshComponent()->SetWorldScale3D(FVector(PlankWidth / 100.0f, 1.0f, 1.0f));
}
```

At this point, the .h file should look like this:

```c++
#pragma once

#include "CoreMinimal.h"
#include "GameFramework/Actor.h"
#include "RopeBridge.generated.h"

class USplineComponent;

USTRUCT(BlueprintType)
struct FCurveParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The alpha parameter of the catenary equation. Defines the shape of the curve."))
	float CatenaryAlpha = 5.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The beta parameter of the catenary equation. Defines the portion of the curve used for the bridge."))
	float CatenaryBeta = 17.5f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The offset of the catenary equation. Defines the horizontal shift of the curve."))
	float CatenaryBetaOffset = 0.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties")
	int SplineSubdivisions = 25;
};

UCLASS()
class ROPESBRIDGE_API ARopeBridge : public AActor
{
	GENERATED_BODY()
	
	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USceneComponent* Root;

	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USplineComponent* Spline;
	
public:	
	ARopeBridge();

protected:
	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	float BridgeLength = 1000.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	FCurveParams CurveParams;

	UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
	void GenerateSplines();

private:
	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	float CalculateChainHeight(float X);

	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	FVector CalculateChainTangent(FVector Previous, FVector Current, FVector Next);

	UPROPERTY(EditAnywhere, Category = "Bridge generation")
	UStaticMesh* PlankMesh;

	UPROPERTY(EditAnywhere, Category = "Bridge generation")
	int32 PlankCount = 25;

	UPROPERTY(EditAnywhere, Category = "Bridge generation")
	float PlankWidth = 100.0f;

	UPROPERTY(EditAnywhere, Category = "Bridge generation")
	float PlankRandomOffset = 50.0f;
};
```

And the .cpp file should look like this:

```c++
#include "RopesBridge.h"
#include "Components/SplineComponent.h"
#include "Components/StaticMeshComponent.h"
#include "Engine/StaticMeshActor.h"
#include "Math/UnrealMathUtility.h"

ARopesBridge::ARopesBridge()
{
	PrimaryActorTick.bCanEverTick = false;

	Root = CreateDefaultSubobject<USceneComponent>(TEXT("Root"));
	RootComponent = Root;

	Spline = CreateDefaultSubobject<USplineComponent>(TEXT("Spline"));
	Spline->SetupAttachment(Root);
}

void ARopesBridge::GenerateSplines()
{
	Spline->ClearSplinePoints(false);
		
	float HeightOffset = CalculateChainHeight(0.0f);
	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Location = FVector(BridgeLength * i / (CurveParams.SplineSubdivisions - 1), 0.0f, CalculateChainHeight(i / (float)(CurveParams.SplineSubdivisions - 1)) - HeightOffset);
		Spline->AddSplinePoint(Location, ESplineCoordinateSpace::Local, false);
	}

	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Tangent = FVector(0.0f, 0.0f, 0.0f);

		if (i != 0 && i < CurveParams.SplineSubdivisions - 1)
		{
			FVector PreviousLocation = Spline->GetLocationAtSplinePoint(i - 1, ESplineCoordinateSpace::Local);
			FVector Location = Spline->GetLocationAtSplinePoint(i, ESplineCoordinateSpace::Local);
			FVector NextLocation = Spline->GetLocationAtSplinePoint(i + 1, ESplineCoordinateSpace::Local);

			Tangent = CalculateChainTangent(PreviousLocation, Location, NextLocation);
		}

	

	Spline->SetTangentAtSplinePoint(i, Tangent, ESplineCoordinateSpace::Local, false);
	}

	Spline->UpdateSpline();

	for (int i = 0; i < PlankCount; i++)
	{
		float Distance = BridgeLength * i / (PlankCount - 1);
		FVector Location = Spline->GetLocationAtDistanceAlongSpline(Distance, ESplineCoordinateSpace::World);
		FRotator Rotation = Spline->GetRotationAtDistanceAlongSpline(Distance, ESplineCoordinateSpace::World);

		float RandomOffset = FMath::RandRange(-PlankRandomOffset, PlankRandomOffset);
		FVector Offset = FVector(0.0f, RandomOffset, 0.0f);

		FActorSpawnParameters SpawnParams;
		AStaticMeshActor* Plank = GetWorld()->SpawnActor<AStaticMeshActor>(Location + Offset, Rotation, SpawnParams);
		Plank->GetStaticMeshComponent()->SetStaticMesh(PlankMesh);
		Plank->GetStaticMeshComponent()->SetWorldScale3D(FVector(PlankWidth / 100.0f, 1.0f, 1.0f));
	}
}

float ARopesBridge::CalculateChainHeight(float X)
{
	return (CurveParams.CatenaryAlpha * cosh(X * CurveParams.CatenaryBeta) + CurveParams.CatenaryBetaOffset);
}

FVector ARopesBridge::CalculateChainTangent(FVector Previous, FVector Current, FVector Next)
{
	FVector Tangent = (Next - Previous).GetSafeNormal();
	return Tangent;
}
```